In [ ]:
from datetime import date
from urllib.parse import quote

import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"

# crashes labeled Bronx, plus crashes with no borough whose coordinates fall in a box around the Bronx.
# blank-borough crashes (many of them on highways like the Major Deegan and Cross Bronx) were being missed; notebook 02's spatial join
# removes anything in the box that's outside the real Bronx
WHERE = (
    "borough='BRONX' OR (borough IS NULL"
    " AND latitude between 40.785 and 40.917"
    " AND longitude between -73.933 and -73.765)"
)

print(pd.read_csv(f"{BASE_URL}?$select=count(*)&$where={quote(WHERE)}"))

    count
0  330480


In [4]:
select_cols = (
    "collision_id,crash_date,crash_time,on_street_name,cross_street_name,off_street_name,"
    "latitude,longitude,number_of_persons_injured,number_of_persons_killed,"
    "contributing_factor_vehicle_1"
)

batch_size = 50000
offset = 0
all_batches = []

while True:
    query = (
        f"?$select={select_cols}"
        f"&$where={quote(WHERE)}"  # quote() because the query has spaces, which urls can't contain
        f"&$order=collision_id"  # fixed order so pages don't overlap or skip rows
        f"&$limit={batch_size}"
        f"&$offset={offset}"
    )
    batch = pd.read_csv(BASE_URL + query)

    if batch.empty:  # stop only when there's nothing left, not on a short batch
        break

    all_batches.append(batch)
    print(f"Pulled {len(batch)} rows (offset {offset})")
    offset += batch_size

df = pd.concat(all_batches, ignore_index=True)
print(f"Total rows pulled: {len(df)}")
print(f"Pulled on {date.today()}")  # the live dataset changes, so record when this snapshot was taken

df.to_csv("../data/raw/bronx_collisions_raw.csv", index=False)

Pulled 50000 rows (offset 0)
Pulled 50000 rows (offset 50000)
Pulled 50000 rows (offset 100000)
Pulled 50000 rows (offset 150000)
Pulled 50000 rows (offset 200000)
Pulled 50000 rows (offset 250000)
Pulled 30480 rows (offset 300000)
Total rows pulled: 330480
Pulled on 2026-11-21
